In [ ]:
from lakehouse_spark import SparkSessionContext
from lakehouse_publish import publish_table


with SparkSessionContext("dota-duration-15min-stage") as ctx:
    spark = ctx

    with ctx.lineage(
        "dota-duration-15min-player-team-map",
        inputs=["raw.dota_duration_15min_match_players"],
        output="stage.dota_duration_15min_player_team_map",
        dataset_descriptions={
            "stage.dota_duration_15min_player_team_map":
                "Соответствие игрока и команды в матче (Radiant/Dire)."
        }
    ):
        teams = spark.sql("""
            select distinct
                match_id,
                account_id,
                case when player_slot < 128 then 'radiant' else 'dire' end as team
            from lakehouse.raw.dota_duration_15min_match_players
            where account_id is not null
              and account_id <> 4294967295
        """)
        teams.writeTo("lakehouse.stage.dota_duration_15min_player_team_map") \
             .using("iceberg").createOrReplace()
        publish_table(
            "stage", "dota_duration_15min_player_team_map", teams,
            description="Команда игрока в матче (Radiant/Dire)",
            dependencies=["raw/dota_duration_15min_match_players"]
        )

    for field in ("attacker", "steam_account_id", "target"):
        cnt = spark.sql(f"""
            select count(*)
            from lakehouse.raw.dota_duration_15min_kill_events k
            join lakehouse.stage.dota_duration_15min_player_team_map t
              on k.match_id = t.match_id and k.{field} = t.account_id
            where k.time between 0 and 899
        """).collect()[0][0]
        print(f"[DIAG] kills 0..899 with {field} matching team_map: {cnt}")

    with ctx.lineage(
        "dota-duration-15min-match-features",
        inputs=[
            "raw.dota_duration_15min_matches",
            "raw.dota_duration_15min_match_players",
            "raw.dota_duration_15min_kill_events",
            "raw.dota_duration_15min_gold_events",
            "raw.dota_duration_15min_tower_death_events",
            "stage.dota_duration_15min_player_team_map",
        ],
        output="stage.dota_duration_15min_match_features",
        dataset_descriptions={
            "stage.dota_duration_15min_match_features":
                "Признаки матча на 15-й минуте + duration. Kills_radiant считается "
                "по лучшему сигналу (attacker → steam_account_id → target-инверсия); "
                "kills_dire = kills_15m − kills_radiant. Башни — просто count(*) за 0..899."
        }
    ):
        features = spark.sql("""
            -- 1) Валидные матчи (>=900 сек, 5×5 игроков)
            with matches as (
                select match_id,
                       max(game_mode)        as game_mode,
                       max(duration)         as duration,
                       max(first_blood_time) as first_blood_time
                from lakehouse.raw.dota_duration_15min_matches
                group by match_id
            ),
            players as (
                select distinct match_id, player_slot, account_id
                from lakehouse.raw.dota_duration_15min_match_players
                where account_id is not null
                  and account_id <> 4294967295
            ),
            player_counts as (
                select match_id,
                       sum(case when player_slot <  128 then 1 else 0 end) as radiant_count,
                       sum(case when player_slot >= 128 then 1 else 0 end) as dire_count
                from players
                group by match_id
            ),
            valid_matches as (
                select m.match_id, m.game_mode, m.duration, m.first_blood_time
                from matches m
                join player_counts pc on pc.match_id = m.match_id
                where m.duration >= 900
                  and pc.radiant_count = 5
                  and pc.dire_count = 5
            ),

            -- 2) Убийства 0..899 и определение команды убийцы
            kills as (
                select distinct match_id, steam_account_id, attacker, target, time
                from lakehouse.raw.dota_duration_15min_kill_events
                where time between 0 and 899
            ),
            kills_with_teams as (
                select
                    k.match_id,
                    ta.team as attacker_team,   -- team of attacker (killer)
                    ts.team as steam_team,      -- team of steam_account_id (fallback killer)
                    tt.team as target_team      -- team of victim
                from kills k
                left join lakehouse.stage.dota_duration_15min_player_team_map ta
                       on k.match_id = ta.match_id and k.attacker = ta.account_id
                left join lakehouse.stage.dota_duration_15min_player_team_map ts
                       on k.match_id = ts.match_id and k.steam_account_id = ts.account_id
                left join lakehouse.stage.dota_duration_15min_player_team_map tt
                       on k.match_id = tt.match_id and k.target = tt.account_id
            ),
            kills_agg as (
                select
                    match_id,
                    count(*) as kills_15m,
                    sum(
                        case
                            -- 1) атакующий однозначно опознан
                            when attacker_team is not null
                                 then case when attacker_team = 'radiant' then 1 else 0 end
                            -- 2) fallback: steam_account_id
                            when steam_team is not null
                                 then case when steam_team = 'radiant' then 1 else 0 end
                            -- 3) fallback: по команде жертвы (dire жертва ⇒ radiant убийца)
                            when target_team = 'dire'    then 1
                            when target_team = 'radiant' then 0
                            -- 4) ничего не знаем — уйдёт в dire при вычитании
                            else 0
                        end
                    ) as kills_radiant_15m
                from kills_with_teams
                group by match_id
            ),

            -- 3) Золото 0..899
            gold as (
                select distinct match_id, steam_account_id, time, amount
                from lakehouse.raw.dota_duration_15min_gold_events
                where time between 0 and 899
            ),
            gold_agg as (
                select g.match_id,
                       sum(case when t.team = 'radiant' then g.amount else 0 end) as gold_radiant_15m,
                       sum(case when t.team = 'dire'    then g.amount else 0 end) as gold_dire_15m,
                       sum(case when t.team = 'radiant' then  g.amount
                                when t.team = 'dire'    then -g.amount
                                else 0 end)                                       as gold_diff_15m
                from gold g
                left join lakehouse.stage.dota_duration_15min_player_team_map t
                       on g.match_id = t.match_id
                      and g.steam_account_id = t.account_id
                group by g.match_id
            ),

            -- 4) Башни 0..899 — просто «сколько упало»
            towers as (
                select distinct match_id, time
                from lakehouse.raw.dota_duration_15min_tower_death_events
                where time between 0 and 899
            ),
            towers_agg as (
                select match_id,
                       min(time) as first_tower_time,
                       count(*)  as total_towers_15m
                from towers
                group by match_id
            )

            -- 5) Финальная сборка
            select
                v.match_id,
                v.game_mode,
                v.duration,
                v.first_blood_time,

                coalesce(tw.first_tower_time, 900)                          as first_tower_time,
                case when tw.first_tower_time is not null then 1 else 0 end as has_tower_15m,

                coalesce(k.kills_15m,         0)                             as kills_15m,
                coalesce(k.kills_radiant_15m, 0)                             as kills_radiant_15m,
                -- вторая сторона — остаток от общего числа
                coalesce(k.kills_15m, 0) - coalesce(k.kills_radiant_15m, 0)  as kills_dire_15m,
                -- diff = radiant - dire = 2*radiant - total
                2 * coalesce(k.kills_radiant_15m, 0) - coalesce(k.kills_15m, 0) as kills_diff_15m,

                coalesce(g.gold_radiant_15m, 0)                              as gold_radiant_15m,
                coalesce(g.gold_dire_15m,    0)                              as gold_dire_15m,
                coalesce(g.gold_diff_15m,    0)                              as gold_diff_15m,
                abs(coalesce(g.gold_diff_15m, 0))                            as abs_gold_diff_15m,

                coalesce(tw.total_towers_15m, 0)                             as total_towers_15m
            from valid_matches v
            left join kills_agg  k  on v.match_id = k.match_id
            left join gold_agg   g  on v.match_id = g.match_id
            left join towers_agg tw on v.match_id = tw.match_id
        """)
        features.writeTo("lakehouse.stage.dota_duration_15min_match_features") \
                .using("iceberg").createOrReplace()
        publish_table(
            "stage", "dota_duration_15min_match_features", features,
            description="Признаки матча на 15-й минуте + duration (цель для линейной регрессии)",
            dependencies=[
                "raw/dota_duration_15min_matches",
                "raw/dota_duration_15min_match_players",
                "raw/dota_duration_15min_kill_events",
                "raw/dota_duration_15min_gold_events",
                "raw/dota_duration_15min_tower_death_events",
                "stage/dota_duration_15min_player_team_map",
            ]
        )

    features.describe().show(truncate=False)

    spark.sql("""
        select
            count(*) as total,
            sum(case when kills_15m        > 0 then 1 else 0 end) as non_zero_total_kills,
            sum(case when kills_radiant_15m> 0 then 1 else 0 end) as non_zero_radiant,
            sum(case when kills_dire_15m   > 0 then 1 else 0 end) as non_zero_dire,
            sum(case when kills_radiant_15m + kills_dire_15m <> kills_15m
                     then 1 else 0 end)                            as broken_sum,
            sum(case when total_towers_15m > 0 then 1 else 0 end) as non_zero_towers
        from lakehouse.stage.dota_duration_15min_match_features
    """).show(truncate=False)

[DIAG] kills 0..899 with attacker matching team_map: 0
[DIAG] kills 0..899 with steam_account_id matching team_map: 373591
[DIAG] kills 0..899 with target matching team_map: 0


+-------+------------------+-----------------+------------------+-----------------+------------------+-------------------+-----------------+------------------+------------------+-------------------+------------------+------------------+-----------------+-----------------+------------------+
|summary|match_id          |game_mode        |duration          |first_blood_time |first_tower_time  |has_tower_15m      |kills_15m        |kills_radiant_15m |kills_dire_15m    |kills_diff_15m     |gold_radiant_15m  |gold_dire_15m     |gold_diff_15m    |abs_gold_diff_15m|total_towers_15m  |
+-------+------------------+-----------------+------------------+-----------------+------------------+-------------------+-----------------+------------------+------------------+-------------------+------------------+------------------+-----------------+-----------------+------------------+
|count  |13048             |13048            |13048             |13048            |13048             |13048              |13

26/11/14 06:30:07 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-a95df88e-ab73-4973-9e9b-049b45c41227/pyspark-17eea17e-1499-4dca-8ee0-02a1caa80409. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-a95df88e-ab73-4973-9e9b-049b45c41227/pyspark-17eea17e-1499-4dca-8ee0-02a1caa80409
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:352)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:269)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:248)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:158)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:157)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1062)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4(ShutdownHookManager.scala:70)
	at org.apache.spark.util.ShutdownHookManager$.$anonfu